In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import matplotlib.pyplot as plt

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
assert X_train.shape == (50000, 32, 32, 3)
assert X_test.shape == (10000, 32, 32, 3)
assert y_train.shape == (50000, 1)
assert y_test.shape == (10000, 1)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    image = tf.image.random_hue(image, max_delta=0.2)
    return image, label

def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1
    image = tf.image.resize(image, (227, 227))
    return image, label

def prepare_dataset(ds, batch_size=100, augment_data=False):
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    ds = ds.shuffle(buffer_size=10000).batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
    return ds

train_ds = prepare_dataset(train_ds, augment_data=True)
test_ds = prepare_dataset(test_ds, augment_data=False)

In [ ]:
model = tf.keras.models.Sequential(
    [
        tf.keras.layers.Conv2D(filters=96, kernel_size=(11, 11), strides=(4, 4), activation='relu', kernel_initializer='he_normal', input_shape=(227, 227, 3)),
        tf.keras.layers.MaxPool2D(pool_size=(3, 3), strides=(2, 2)),
        tf.keras.layers.Conv2D(filters=256, kernel_size=(5, 5), strides=(1, 1), activation='relu', kernel_initializer='he_normal', padding='same'),
        tf.keras.layers.MaxPool2D(pool_size=(3, 3), strides=(2, 2)),
        tf.keras.layers.Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), activation='relu', kernel_initializer='he_normal', padding='same'),
        tf.keras.layers.Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), activation='relu', kernel_initializer='he_normal', padding='same'),
        tf.keras.layers.Conv2D(filters=256, kernel_size=(3, 3), strides=(1, 1), activation='relu', kernel_initializer='he_normal', padding='same'),
        tf.keras.layers.MaxPool2D(pool_size=(3, 3), strides=(2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(4096, activation='relu', kernel_initializer='he_normal'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(4096, activation='relu', kernel_initializer='he_normal'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10, activation='softmax'),
    ]
)

model.compile(
    optimizer=tf.keras.optimizers.SGD(momentum=0.9, learning_rate=0.01, nesterov=True),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 55, 55, 96)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 27, 27, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 27, 27, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 13, 13, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 13, 13, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 13, 13, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │        40,970 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,322,314 (222.48 MB)

 Trainable params: 58,322,314 (222.48 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=3, min_lr=0.001)
early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
callbacks = [early_stopping, reduce_lr]

history = model.fit(train_ds, epochs=20, validation_data=test_ds, batch_size=100, callbacks=callbacks)

Epoch 1/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 109s 171ms/step - accuracy: 0.2602 - loss: 2.0475 - val_accuracy: 0.5797 - val_loss: 1.1801 - learning_rate: 0.0100
Epoch 2/20
 14/500 ━━━━━━━━━━━━━━━━━━━━ 1:06 138ms/step - accuracy: 0.4947 - loss: 1.3727

In [ ]:
plt.figure(fig_size=(12, 4))
plt.subplots(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')

plt.subplots(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import tf_keras

from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import matplotlib.pyplot as plt

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
assert X_train.shape == (50000, 32, 32, 3)
assert X_test.shape == (10000, 32, 32, 3)
assert y_train.shape == (50000, 1)
assert y_test.shape == (10000, 1)

In [ ]:
# Convert the data to TensorFlow Datasets
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Preprocessing function
def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = (image / 255.0)
    image = tf.image.resize(image, (224,224))
    return image, label

# Dataset preparation function with augmentation option
def prepare_dataset(ds, batch_size=20):
    ds = ds.map(preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    ds = ds.shuffle(buffer_size=10000).batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
    return ds

# Preprocess the data
train_ds = prepare_dataset(train_ds)
test_ds = prepare_dataset(test_ds)

In [ ]:
model_url = "https://www.kaggle.com/models/google/efficientnet-v2/TensorFlow2/imagenet1k-b0-feature-vector/2"

model = tf_keras.Sequential([
    tf_keras.layers.Input(shape=(224, 224, 3)),
    hub.KerasLayer(model_url, trainable=False, name="feature_extraction_layer"),
    tf_keras.layers.Dropout(0.2),
    tf_keras.layers.Dense(128, activation="relu", kernel_initializer="he_normal"),
    tf_keras.layers.Dropout(0.2),
    tf_keras.layers.Dense(10, activation="softmax"),
])

model.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf_keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

In [ ]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 feature_extraction_layer (  (None, 1280)              5919312   
 KerasLayer)                                                     
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 128)               163968    
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 10)                1290      
                                                                 
Total params: 6084570 (23.21 MB)
Trainable params: 165258 (645.54 KB)
Non-trainable params: 5919312 (22.58 MB)
___________

In [ ]:
# Train the model
history = model.fit(
    train_ds, epochs=100, validation_data=test_ds, batch_size=20)

NameError: name 'model' is not defined

In [1]:
import tensorflow as tf

In [2]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
assert X_train.shape == (50000, 32, 32, 3)
assert X_test.shape == (10000, 32, 32, 3)
assert y_train.shape == (50000, 1)
assert y_test.shape == (10000, 1)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [3]:
def preprocess_image_input(input_images):
    input_images = input_images.astype('float32')
    output_ims = tf.keras.applications.xception.preprocess_input(input_images)
    return output_ims


X_train = preprocess_image_input(X_train)
X_test = preprocess_image_input(X_test)

In [4]:
inputs = tf.keras.layers.Input(shape=(32, 32, 3), name='input_layer')
resize = tf.keras.layers.UpSampling2D(size=(7, 7), name='resize_layer')(inputs)
feature_extractor = tf.keras.applications.Xception(input_shape=(224, 224, 3),
                               include_top=False,
                               weights='imagenet',)(resize)
x = tf.keras.layers.GlobalAveragePooling2D(name='global_avg_pool_layer')(feature_extractor)
x = tf.keras.layers.Flatten(name='flatten_layer')(x)
x = tf.keras.layers.Dense(1024, activation='relu', name='dense_layer_1', kernel_initializer='he_normal')(x)
x = tf.keras.layers.Dense(512, activation='relu', name='dense_layer_2', kernel_initializer='he_normal')(x)
classification_output = tf.keras.layers.Dense(10, activation='softmax', name='output_layer')(x)

model = tf.keras.Model(inputs, outputs=classification_output, name='my_model')

model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

model.summary()

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "my_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resize_layer (UpSampling2D)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ xception (Functional)           │ (None, 7, 7, 2048)     │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_avg_pool_layer           │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_layer (Flatten)         │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_layer_1 (Dense)           │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_layer_2 (Dense)           │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,489,586 (89.61 MB)

 Trainable params: 23,435,058 (89.40 MB)

 Non-trainable params: 54,528 (213.00 KB)

In [5]:
history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 939s 558ms/step - accuracy: 0.7451 - loss: 0.8020 - val_accuracy: 0.8213 - val_loss: 0.5957
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 834s 533ms/step - accuracy: 0.8932 - loss: 0.3284 - val_accuracy: 0.8568 - val_loss: 0.4369
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 835s 534ms/step - accuracy: 0.9282 - loss: 0.2286 - val_accuracy: 0.8914 - val_loss: 0.3414
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 861s 533ms/step - accuracy: 0.9427 - loss: 0.1761 - val_accuracy: 0.8996 - val_loss: 0.3272
Epoch 5/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 834s 534ms/step - accuracy: 0.9536 - loss: 0.1423 - val_accuracy: 0.9091 - val_loss: 0.3069


In [6]:
len(model.layers[2].layers)

132

In [7]:
for layer in model.layers[2].layers[:100]:
    layer.trainable = False # Freeze the layers

In [8]:
model.fit(X_train, y_train, epochs=2, validation_data=(X_test, y_test))

Epoch 1/2
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 833s 533ms/step - accuracy: 0.9628 - loss: 0.1144 - val_accuracy: 0.8921 - val_loss: 0.3834
Epoch 2/2
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 833s 533ms/step - accuracy: 0.9695 - loss: 0.0942 - val_accuracy: 0.9030 - val_loss: 0.3482
